# SHMS — Phase 3A (LSTM) & 3C (GNN) Training
**Data: Google Drive** | **Compute: Colab T4 GPU**

Path Drive: `My Drive/Penelitian/shms-ai-anomaly-detection/02_data/processed/`

| | Path |
|---|---|
| Data .npy | `/content/drive/MyDrive/Penelitian/shms-ai-anomaly-detection/02_data/processed/` |
| Model output | `/content/models/` (Temporary — di-download di akhir) |
| Code | `/content/code/` (Temporary) |

## ① Verifikasi GPU & Mount Google Drive

In [ ]:
import torch, os, sys, shutil, time
import numpy as np
from pathlib import Path

# --- Cek GPU ---
print('=== ENVIRONMENT CHECK ===')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'GPU RAM  : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
else:
    print('⚠️  Tidak ada GPU! Runtime → Change runtime type → T4 GPU')
    raise RuntimeError('GPU diperlukan untuk training yang efisien')

In [ ]:
# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Verifikasi path data
GDRIVE_PROCESSED = Path('/content/drive/MyDrive/Penelitian/shms-ai-anomaly-detection/02_data/processed')

if not GDRIVE_PROCESSED.exists():
    raise FileNotFoundError(
        f'Path tidak ditemukan: {GDRIVE_PROCESSED}\n'
        f'Pastikan folder sudah ada di Google Drive.'
    )

npy_count = len(list(GDRIVE_PROCESSED.glob('*_X.npy')))
csv_count = len(list(GDRIVE_PROCESSED.glob('*.csv')))

print(f'\n✅ Google Drive terhubung')
print(f'   Path    : {GDRIVE_PROCESSED}')
print(f'   .npy    : {npy_count} file (_X.npy)')
print(f'   .csv    : {csv_count} file')

# Cek file wajib ada
required = ['processing_summary.csv', 'p2_normalizer_stats.csv']
for f in required:
    p = GDRIVE_PROCESSED / f
    status = '✅' if p.exists() else '❌ TIDAK ADA'
    print(f'   {status}  {f}')

In [ ]:
# Diagnostik: verifikasi normalisasi dan kondisi data
import pandas as pd, numpy as np

norm_path = GDRIVE_PROCESSED / 'p2_normalizer_stats.csv'
norm_df = pd.read_csv(str(norm_path))
mask = norm_df['channel'].str.contains('CA_L02|CA_L17|AC_PY1T', na=False)
print('Normalizer stats:')
print(norm_df[mask].to_string(index=False))

summary = pd.read_csv(str(GDRIVE_PROCESSED / 'processing_summary.csv'))
day1 = str(summary[summary['split']=='train']['date'].iloc[0])
xp = GDRIVE_PROCESSED / f'{day1}_X.npy'
with open(str(xp), 'rb') as f:
    X_s = np.load(f)

print(f'\nSample {day1}_X.npy:')
print(f'  shape = {X_s.shape}')
print(f'  mean  = {X_s.mean():.3f}')
print(f'  std   = {X_s.std():.3f}')
print(f'  max   = {X_s.max():.1f}')

if X_s.max() > 100:
    print('\n>>> Data RAW (belum normalized)')
    print('>>> Normalisasi diterapkan di dalam __iter__ — loss awal 10-100 NORMAL')
    print('>>> Loss seharusnya stabil di epoch 5-10 jika normalisasi aktif')
else:
    print('\n>>> Data sudah normalized — loss awal seharusnya < 2')


## ② Buat Folder Sementara & Upload Kode Python

In [ ]:
# Folder temporary di /content/ (bukan Drive)
# Model disimpan di sini dulu, baru dipindah ke Drive di akhir
DIRS = {
    'code'   : Path('/content/code'),
    'models' : Path('/content/models'),
    'results': Path('/content/results/figures'),
}
for name, p in DIRS.items():
    p.mkdir(parents=True, exist_ok=True)
    print(f'✅ /content/{name}/')

In [ ]:
# Upload file kode Python dari laptop
# File yang dibutuhkan:
#   shms_config.py
#   shms_phase2_preprocessing.py
#   shms_phase3a_lstm.py
#   shms_phase3c_gnn.py

from google.colab import files
print('Upload 4 file Python (pilih semua sekaligus):')
uploaded = files.upload()

for fname in uploaded:
    dst = f'/content/code/{fname}'
    shutil.move(fname, dst)
    print(f'  ✅ {fname}')

sys.path.insert(0, '/content/code')
print('\nsys.path updated')

## ③ Patch shms_config.py → arahkan ke Google Drive

In [ ]:
CONFIG_PATH = '/content/code/shms_config.py'

COLAB_PATCH = f'''

# ══════════════════════════════════════════════════════
# COLAB + GOOGLE DRIVE PATCH — ditambahkan otomatis
# Data .npy dibaca langsung dari Google Drive (tidak di-copy)
# Model output ke /content/models/ (temporary)
# ══════════════════════════════════════════════════════
from pathlib import Path as _P

# Data .npy → langsung dari Drive (tidak perlu copy)
_GDRIVE = _P('/content/drive/MyDrive/Penelitian/shms-ai-anomaly-detection/02_data/processed')

# Model dan hasil → temporary storage (di-download di akhir)
MODEL_DIR          = _P('/content/models')
RESULTS_DIR        = _P('/content/results')
DATA_PROCESSED_DIR = _GDRIVE
OUT_DATA_OVERRIDE  = _GDRIVE
PROJECT_ROOT       = _P('/content')

def get_processed_dir():
    return _GDRIVE

def get_raw_dir():
    return _GDRIVE

def get_abnormal_dir():
    return _GDRIVE
'''

with open(CONFIG_PATH, 'a') as f:
    f.write(COLAB_PATCH)

# Reload & verifikasi
for mod in list(sys.modules.keys()):
    if 'shms' in mod:
        del sys.modules[mod]

import shms_config, importlib
importlib.reload(shms_config)

print('✅ Patch berhasil')
print(f'  get_processed_dir() → {shms_config.get_processed_dir()}')
print(f'  MODEL_DIR           → {shms_config.MODEL_DIR}')
print(f'  RESULTS_DIR         → {shms_config.RESULTS_DIR}')
print(f'  MAIN_CHANNELS       → {len(shms_config.MAIN_CHANNELS)} channel')

# Buat folder output
shms_config.MODEL_DIR.mkdir(parents=True, exist_ok=True)
shms_config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(shms_config.RESULTS_DIR / 'figures').mkdir(parents=True, exist_ok=True)

## ④ Verifikasi Data di Drive

In [ ]:
import pandas as pd

summary_path = GDRIVE_PROCESSED / 'processing_summary.csv'
summary = pd.read_csv(summary_path)

print('=== VERIFIKASI DATA ===')
print(f'Total hari terdaftar : {len(summary)}')
for split in ['train', 'val', 'test']:
    n = (summary['split'] == split).sum()
    print(f'  {split:<6}: {n} hari')
print()

# Cek file .npy yang ada vs yang dibutuhkan
needed_days = summary[
    summary['split'].isin(['train', 'val']) &
    (summary['status'] == 'ok')
]['date'].astype(str).tolist()

available = {f.stem.replace('_X', '') for f in GDRIVE_PROCESSED.glob('*_X.npy')}
needed    = set(needed_days)
missing   = needed - available
ok_count  = needed & available

print(f'File .npy dibutuhkan   : {len(needed)} hari (train + val)')
print(f'File .npy tersedia     : {len(ok_count)} hari')
print(f'File .npy tidak ada    : {len(missing)} hari')

if missing:
    print('\n⚠️  File yang belum ada di Drive:')
    for d in sorted(missing):
        print(f'   {d}_X.npy')
else:
    print('\n✅ Semua file tersedia — siap training!')

# Estimasi total size
total_gb = sum(
    (GDRIVE_PROCESSED / f'{d}_X.npy').stat().st_size
    for d in ok_count
    if (GDRIVE_PROCESSED / f'{d}_X.npy').exists()
) / 1024**3
print(f'\nTotal ukuran data : {total_gb:.1f} GB')

## ⑤ Phase 3A — Training LSTM Autoencoder

> **Catatan penting**: Data dibaca langsung dari Google Drive per file per epoch.
> Kecepatan baca Drive ~100 MB/s. Bottleneck ada di I/O, bukan GPU.
> Jika terlalu lambat, gunakan sel opsional di bawah untuk copy ke `/content/` dulu.

In [ ]:
# [OPSIONAL] Copy data dari Drive ke /content/ untuk I/O lebih cepat
# Jalankan ini hanya jika training terasa lambat karena I/O Drive
# Butuh ~46 GB temporary storage di /content/

COPY_TO_LOCAL = False  # Ubah ke True jika mau copy ke /content/

if COPY_TO_LOCAL:
    LOCAL_DIR = Path('/content/processed_local')
    LOCAL_DIR.mkdir(exist_ok=True)

    files_to_copy = list(GDRIVE_PROCESSED.glob('*_X.npy')) + \
                    list(GDRIVE_PROCESSED.glob('*_y.npy')) + \
                    list(GDRIVE_PROCESSED.glob('*.csv'))

    print(f'Copying {len(files_to_copy)} files ke /content/processed_local/ ...')
    t0 = time.time()
    for i, src in enumerate(files_to_copy):
        dst = LOCAL_DIR / src.name
        if not dst.exists():
            shutil.copy2(str(src), str(dst))
        if (i+1) % 10 == 0:
            print(f'  {i+1}/{len(files_to_copy)} files ({time.time()-t0:.0f}s)')

    # Override config ke local path
    import shms_config
    shms_config.DATA_PROCESSED_DIR = LOCAL_DIR
    shms_config.OUT_DATA_OVERRIDE  = LOCAL_DIR
    shms_config.get_processed_dir  = lambda: LOCAL_DIR
    print(f'\n✅ Copy selesai ({time.time()-t0:.0f}s)')
    print(f'   get_processed_dir() → {shms_config.get_processed_dir()}')
else:
    print('Skip copy — baca langsung dari Drive')
    print(f'get_processed_dir() → {shms_config.get_processed_dir()}')

In [ ]:
# Load module LSTM
for mod in list(sys.modules.keys()):
    if 'shms' in mod:
        del sys.modules[mod]
sys.path.insert(0, '/content/code')

import shms_config, importlib
importlib.reload(shms_config)
import shms_phase3a_lstm as p3a
importlib.reload(p3a)

print('=== KONFIGURASI LSTM ===')
print(f'Device          : {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'Data dir        : {shms_config.get_processed_dir()}')
print(f'Model dir       : {shms_config.MODEL_DIR}')
print(f'Main channels   : {len(shms_config.MAIN_CHANNELS)}')

In [ ]:
HP_LSTM = {
    **p3a.HP,
    'batch_size'        : 256,
    'max_train_windows' : 100_000,
    'n_epochs'          : 100,
    'patience'          : 15,
    'learning_rate'     : 3e-4,
    'clip_grad'         : 0.5,
}

print('Hyperparameter LSTM:')
for k, v in HP_LSTM.items():
    orig = p3a.HP.get(k, '-')
    diff = ' <- diubah' if v != orig else ''
    print(f'  {k:<22}: {str(v):<12} {diff}')


In [ ]:
# === TRAINING LSTM ===
t_start = time.time()

model_lstm, history_lstm = p3a.train(
    hp          = HP_LSTM,
    save_dir    = shms_config.MODEL_DIR,
    results_dir = shms_config.RESULTS_DIR,
)

elapsed = time.time() - t_start
print(f'\n✅ LSTM Training selesai: {elapsed/60:.1f} menit')

In [ ]:
# Kalibrasi threshold LSTM
threshold_lstm = p3a.calibrate_threshold(
    model       = model_lstm,
    hp          = HP_LSTM,
    save_dir    = shms_config.MODEL_DIR,
    results_dir = shms_config.RESULTS_DIR,
)
print(f'\n✅ Threshold LSTM (P{HP_LSTM["threshold_pct"]}): {threshold_lstm:.6f}')

## ⑥ Phase 3C — Training GNN Autoencoder

In [ ]:
# Install PyTorch Geometric
import subprocess
torch_ver = torch.__version__.split('+')[0]
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'

print(f'Installing PyG untuk torch-{torch_ver}+{cuda_tag} ...')
result = subprocess.run(
    ['pip', 'install', '-q', 'torch-geometric'],
    capture_output=True, text=True
)
print('✅ torch-geometric installed' if result.returncode == 0 else f'❌ {result.stderr[-200:]}')

In [ ]:
# Load GNN module
for mod in list(sys.modules.keys()):
    if 'shms' in mod:
        del sys.modules[mod]

import shms_config, importlib
importlib.reload(shms_config)
import shms_phase3c_gnn as p3c
importlib.reload(p3c)

print('=== KONFIGURASI GNN ===')
print(f'PyG available   : {p3c.PYG_AVAILABLE}')
print(f'Torch available : {p3c.TORCH_AVAILABLE}')
print(f'N nodes (sensor): {p3c.N_NODES}')
print(f'Data dir        : {shms_config.get_processed_dir()}')

In [ ]:
HP_GNN = {
    **p3c.HP,
    'batch_size'        : 256,
    'max_train_windows' : 100_000,
    'n_epochs'          : 100,
    'patience'          : 15,
    'learning_rate'     : 3e-4,
    'clip_grad'         : 0.5,
}

print('Hyperparameter GNN:')
for k, v in HP_GNN.items():
    orig = p3c.HP.get(k, '-')
    diff = ' <- diubah' if v != orig else ''
    print(f'  {k:<22}: {str(v):<12} {diff}')


In [ ]:
# === TRAINING GNN ===
t_start = time.time()

model_gnn, graph_data, history_gnn = p3c.train(
    hp          = HP_GNN,
    save_dir    = shms_config.MODEL_DIR,
    results_dir = shms_config.RESULTS_DIR,
)

elapsed = time.time() - t_start
print(f'\n✅ GNN Training selesai: {elapsed/60:.1f} menit')

In [ ]:
# Kalibrasi threshold GNN
threshold_gnn = p3c.calibrate_threshold(
    model       = model_gnn,
    graph_data  = graph_data,
    hp          = HP_GNN,
    save_dir    = shms_config.MODEL_DIR,
    results_dir = shms_config.RESULTS_DIR,
)
print(f'\n✅ Threshold GNN (P{HP_GNN["threshold_pct"]}): {threshold_gnn:.6f}')

## ⑦ Simpan Model ke Google Drive + Download ke Laptop

Model disimpan ke dua tempat:
1. **Google Drive** — permanen, tidak hilang saat session berakhir
2. **Download** ke laptop — untuk langsung dipakai di ZBook

In [ ]:
import zipfile
from google.colab import files as colab_files

# Folder tujuan di Drive
GDRIVE_MODELS = Path('/content/drive/MyDrive/Penelitian/shms-ai-anomaly-detection/04_models')
GDRIVE_RESULTS = Path('/content/drive/MyDrive/Penelitian/shms-ai-anomaly-detection/05_results')
GDRIVE_MODELS.mkdir(parents=True, exist_ok=True)
GDRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
(GDRIVE_RESULTS / 'figures').mkdir(exist_ok=True)

print('=== MENYIMPAN KE GOOGLE DRIVE ===')

# Daftar file model yang akan disimpan
model_files = [
    'lstm_autoencoder_best.pt',
    'lstm_threshold.json',
    'lstm_hp.json',
    'gnn_model_best.pt',
    'gnn_threshold.json',
    'gnn_hp.json',
    'gnn_graph_data.pkl',
    'iforest_threshold.json',   # sudah ada dari ZBook
]

saved_to_drive = []
for fname in model_files:
    src = shms_config.MODEL_DIR / fname
    if src.exists():
        dst = GDRIVE_MODELS / fname
        shutil.copy2(str(src), str(dst))
        saved_to_drive.append(str(src))
        size_kb = src.stat().st_size / 1024
        print(f'  ✅ {fname} ({size_kb:.0f} KB) → Drive/04_models/')
    else:
        print(f'  ⏭  {fname} tidak ada (skip)')

# Copy figures ke Drive
fig_count = 0
for fig in (shms_config.RESULTS_DIR / 'figures').glob('*.png'):
    shutil.copy2(str(fig), str(GDRIVE_RESULTS / 'figures' / fig.name))
    fig_count += 1
print(f'  ✅ {fig_count} figures → Drive/05_results/figures/')

print(f'\n✅ Semua tersimpan di Google Drive')

In [ ]:
# Kemas semua model dalam 1 zip untuk download ke laptop
zip_path = '/content/shms_models_output.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:

    # Models
    for fname in model_files:
        src = shms_config.MODEL_DIR / fname
        if src.exists():
            zf.write(str(src), f'04_models/{fname}')

    # Figures
    for fig in (shms_config.RESULTS_DIR / 'figures').glob('*.png'):
        zf.write(str(fig), f'05_results/figures/{fig.name}')

    # Metrics CSV
    for csv in shms_config.RESULTS_DIR.glob('*.csv'):
        zf.write(str(csv), f'05_results/{csv.name}')

zip_mb = Path(zip_path).stat().st_size / 1024**2
print(f'ZIP: {zip_mb:.1f} MB — mendownload ke laptop...')
colab_files.download(zip_path)

## ⑧ Setelah Selesai — Langkah di ZBook

Extract `shms_models_output.zip` lalu copy ke folder ZBook:

```
04_models\lstm_autoencoder_best.pt  → D:\...\shms_03_code_zbook\04_models\
04_models\lstm_threshold.json       → D:\...\shms_03_code_zbook\04_models\
04_models\gnn_model_best.pt         → D:\...\shms_03_code_zbook\04_models\
04_models\gnn_threshold.json        → D:\...\shms_03_code_zbook\04_models\
04_models\gnn_graph_data.pkl        → D:\...\shms_03_code_zbook\04_models\
05_results\figures\*.png            → D:\...\shms_03_code_zbook\05_results\figures\
```

Kemudian lanjutkan Phase 4 & 5 di ZBook:
```
python run_pipeline.py --phase 4 5
```

---
**Model juga sudah tersimpan permanen di Google Drive:**
`My Drive/Penelitian/shms-ai-anomaly-detection/04_models/`